# Imports

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from timeit import default_timer as timer
from pathlib import Path
import json
import csv
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
)
from tqdm.notebook import tqdm

# MONAI imports
from monai.data import Dataset
from monai.transforms import (
    Compose, 
    LoadImaged, 
    EnsureChannelFirstd,
    Orientationd, 
    Spacingd, 
    CenterSpatialCropd,
    ResizeD,
    ScaleIntensityd,
    ToTensord,
    RandFlipd,
    RandRotated,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandAffined
)
from monai.networks.nets import DenseNet201

# Configureations

In [2]:
# Configuration settings
CONFIG = {
    'seed': 42,
    'n_folds': 5,
    'batch_size': 8,
    'num_workers': 8,
    'num_epochs': 20,
    'learning_rate': 1e-4,
    'roi_size': (36, 36, 36),
    'resize_d': (128, 128, 128),
    'target_size': (36, 36, 36),
    'train_dir': 'data/train',
    'test_dir': 'data/test',
    'class_names': ['CN', 'AD'],  # CN: 0, AD: 1
    'model_save_dir': 'models',
    'results_dir': 'results',
    'graphs_dir': 'graphs'
}

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create directories if they don't exist
os.makedirs(CONFIG['model_save_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)
os.makedirs(CONFIG['graphs_dir'], exist_ok=True)

# Create timestamp for this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"densenet201_{timestamp}"  # Change this based on the model you're using

# Global variable to track best model performance across folds
best_fold_metrics = {'fold_idx': -1, 'val_acc': 0, 'val_loss': float('inf')}

Using device: cuda


In [3]:
def create_data_dicts(data_dir):
    """Create a list of dictionaries for each image with its path, label, and patient ID."""
    data_dicts = []
    
    # For each class
    for class_idx, class_name in enumerate(CONFIG['class_names']):
        class_dir = os.path.join(data_dir, class_name)
        
        # Get all files in the class directory
        files = [os.path.join(class_dir, fname) for fname in os.listdir(class_dir)
                if os.path.isfile(os.path.join(class_dir, fname))]
        
        # Create a dictionary for each file
        for file_path in files:
            # Extract patient ID from filename
            filename = os.path.basename(file_path)
            # Extract pattern like "002_S_0413" from the filename
            patient_id_match = re.search(r'(\d{3}_S_\d{4})', filename)
            patient_id = patient_id_match.group(1) if patient_id_match else "unknown"
            
            data_dicts.append({
                "image": file_path,
                "label": class_idx,  # CN: 0, AD: 1
                "patient_id": patient_id
            })
    
    return data_dicts

# Patient Distribution

In [4]:
def analyze_patient_distribution():
    """Analyze and save the distribution of patients across datasets."""
    # Get all data dictionaries
    train_dicts = create_data_dicts(CONFIG['train_dir'])
    test_dicts = create_data_dicts(CONFIG['test_dir'])
    
    # Extract patient IDs and their labels
    train_patients = {}
    for d in train_dicts:
        pid = d['patient_id']
        label = d['label']
        if pid not in train_patients:
            train_patients[pid] = {'label': label, 'count': 0}
        train_patients[pid]['count'] += 1
    
    test_patients = {}
    for d in test_dicts:
        pid = d['patient_id']
        label = d['label']
        if pid not in test_patients:
            test_patients[pid] = {'label': label, 'count': 0}
        test_patients[pid]['count'] += 1
    
    # Check for overlap
    train_pids = set(train_patients.keys())
    test_pids = set(test_patients.keys())
    overlap = train_pids.intersection(test_pids)
    
    # Save distribution to CSV
    train_df = pd.DataFrame([
        {'patient_id': pid, 'dataset': 'train', 'label': info['label'], 'scan_count': info['count']}
        for pid, info in train_patients.items()
    ])
    
    test_df = pd.DataFrame([
        {'patient_id': pid, 'dataset': 'test', 'label': info['label'], 'scan_count': info['count']}
        for pid, info in test_patients.items()
    ])
    
    # Combine and save
    all_df = pd.concat([train_df, test_df])
    all_df.to_csv(os.path.join(CONFIG['results_dir'], f"{run_name}_patient_distribution.csv"), index=False)
    
    # Print summary
    print(f"Total unique patients in training: {len(train_patients)}")
    print(f"Total unique patients in testing: {len(test_patients)}")
    print(f"Patient overlap between train and test: {len(overlap)}")
    print(f"Patient distribution saved to {os.path.join(CONFIG['results_dir'], f'{run_name}_patient_distribution.csv')}")
    
    return train_patients, test_patients

# Initialize Model

In [5]:
def initialize_model():
    """Initialize the model architecture."""
    # Initialize DenseNet201
    model = DenseNet201(
        spatial_dims=3,         # 3D model
        in_channels=1,          # Single channel input
        out_channels=2,         # Binary classification (2 classes)
        init_features=64,       # Default initial feature count
        growth_rate=32,         # Default growth rate
        block_config=(6, 12, 48, 32),  # Default block configuration for DenseNet201
    ).to(device)
    
    return model

# Train Epoch

In [6]:
def train_epoch(model, dataloader, optimizer, loss_function, epoch):
    """Train the model for one epoch."""
    model.train()
    epoch_loss = 0
    step = 0
    y_pred = []
    y_true = []
    
    for batch in tqdm(dataloader, desc=f"Training epoch {epoch+1}", unit="batch"):
        step += 1
        inputs, labels = batch["image"].to(device), batch["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Get predictions
        _, predicted = torch.max(outputs, 1)
        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(labels.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    return epoch_loss / step, accuracy, precision, recall, f1

# Validate Epoch

In [7]:
def validate_epoch(model, dataloader, loss_function, epoch):
    """Validate the model on the validation set."""
    model.eval()
    epoch_loss = 0
    step = 0
    y_pred = []
    y_true = []
    y_scores = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Validation epoch {epoch+1}", unit="batch"):
            step += 1
            inputs, labels = batch["image"].to(device), batch["label"].to(device)
            
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            
            epoch_loss += loss.item()
            
            # Get predictions and scores
            probs = torch.softmax(outputs, dim=1)
            scores = probs[:, 1].cpu().numpy()  # Probability for class 1 (AD)
            _, predicted = torch.max(outputs, 1)
            
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(labels.cpu().numpy())
            y_scores.extend(scores)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='binary')
    recall = recall_score(y_true, y_pred, average='binary')
    f1 = f1_score(y_true, y_pred, average='binary')
    
    return epoch_loss / step, accuracy, precision, recall, f1, y_true, y_scores

# Run CV 5-Fold

In [9]:
def run_cross_validation():
    """Run 5-fold cross-validation on the training data, ensuring patient-level separation."""
    global best_fold_metrics
    
    print(f"Starting {CONFIG['n_folds']}-fold cross-validation...")
    start_time = timer()
    
    # Define transforms
    transforms = Compose([
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        Orientationd(keys=["image"], axcodes="RAS"),
        Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
        #CenterSpatialCropd(keys=["image"], roi_size=CONFIG['roi_size']),
        ResizeD(keys=["image"], spatial_size=CONFIG['resize_d'], mode="nearest"),

        # Add augmentations here
        #RandFlipd(keys=["image"], prob=0.5, spatial_axis=None),  # flips across all axes with 50% probability
        #RandRotated(keys=["image"], prob=1.0, range_x=0.26, range_y=0.26, range_z=0.26, mode="bilinear"),  # ±15° rotation (0.26 radians)
        #RandScaleIntensityd(keys=["image"], factors=0.1, prob=1.0),  # Random intensity scaling
        #RandShiftIntensityd(keys=["image"], offsets=0.1, prob=1.0),  # Random intensity shift
        #RandAffined(keys=["image"], prob=0.5, scale_range=(0.1, 0.1, 0.1), mode="bilinear"),  # Random scaling
        
        ScaleIntensityd(keys=["image"]),
        ToTensord(keys=["image"]),
    ])

    val_transforms = Compose([
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        Orientationd(keys=["image"], axcodes="RAS"),
        Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
        #CenterSpatialCropd(keys=["image"], roi_size=CONFIG['roi_size']),
        ResizeD(keys=["image"], spatial_size=CONFIG['resize_d'], mode="nearest"),
        ScaleIntensityd(keys=["image"]),
        ToTensord(keys=["image"]),
    ])
    
    # Create data dictionaries
    data_dicts = create_data_dicts(CONFIG['train_dir'])
    
    # Group data by patient ID
    patients = {}
    for idx, d in enumerate(data_dicts):
        patient_id = d["patient_id"]
        if patient_id not in patients:
            patients[patient_id] = []
        patients[patient_id].append(idx)
    
    # Print dataset statistics
    print(f"Total samples in training dataset: {len(data_dicts)}")
    print(f"Total unique patients: {len(patients)}")
    
    class_counts = {}
    patient_class_counts = {}
    for d in data_dicts:
        label = d['label']
        class_counts[label] = class_counts.get(label, 0) + 1
        
        # Count unique patients per class
        if label not in patient_class_counts:
            patient_class_counts[label] = set()
        patient_class_counts[label].add(d['patient_id'])
    
    for label, count in class_counts.items():
        class_name = CONFIG['class_names'][label]
        n_patients = len(patient_class_counts[label])
        print(f"Class {class_name} (label {label}): {count} samples from {n_patients} unique patients")
    
    # Create patient-level k-fold split
    patient_ids = list(patients.keys())
    kf = KFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])
    
    # Storage for results across all folds
    all_fold_results = []
    fold_best_metrics = []  # Track each fold's best metrics separately
    best_fold_metrics = {'fold_idx': -1, 'val_acc': 0, 'val_loss': float('inf')}
    
    # Run each fold
    for fold_idx, (train_patient_indices, val_patient_indices) in enumerate(kf.split(patient_ids)):
        # Get patient IDs for this split
        train_patients = [patient_ids[i] for i in train_patient_indices]
        val_patients = [patient_ids[i] for i in val_patient_indices]
        
        # Get scan indices for this split
        train_indices = [idx for patient in train_patients for idx in patients[patient]]
        val_indices = [idx for patient in val_patients for idx in patients[patient]]
        
        print(f"\nTraining Fold {fold_idx+1}/{CONFIG['n_folds']}")
        print(f"Train size: {len(train_indices)} scans from {len(train_patients)} patients")
        print(f"Validation size: {len(val_indices)} scans from {len(val_patients)} patients")
        
        # Check class distribution in each split
        train_class_counts = {}
        val_class_counts = {}
        
        for idx in train_indices:
            label = data_dicts[idx]['label']
            train_class_counts[label] = train_class_counts.get(label, 0) + 1
            
        for idx in val_indices:
            label = data_dicts[idx]['label']
            val_class_counts[label] = val_class_counts.get(label, 0) + 1
        
        print("Train class distribution:")
        for label, count in train_class_counts.items():
            class_name = CONFIG['class_names'][label]
            print(f"  {class_name}: {count} samples ({count/len(train_indices)*100:.1f}%)")
            
        print("Validation class distribution:")
        for label, count in val_class_counts.items():
            class_name = CONFIG['class_names'][label]
            print(f"  {class_name}: {count} samples ({count/len(val_indices)*100:.1f}%)")
        
        # Prepare data loaders for this fold
        train_files = [data_dicts[i] for i in train_indices]
        val_files = [data_dicts[i] for i in val_indices]
        
        train_ds = Dataset(data=train_files, transform=transforms)
        val_ds = Dataset(data=val_files, transform=val_transforms)
        
        train_loader = DataLoader(
            train_ds,
            batch_size=CONFIG['batch_size'],
            shuffle=True,
            num_workers=CONFIG['num_workers'],
            pin_memory=torch.cuda.is_available()
        )
        
        val_loader = DataLoader(
            val_ds,
            batch_size=CONFIG['batch_size'],
            shuffle=False,
            num_workers=CONFIG['num_workers'],
            pin_memory=torch.cuda.is_available()
        )
        
        # Initialize model, loss function, and optimizer
        model = initialize_model()
        loss_function = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
        
        # Training and validation history
        train_losses = []
        val_losses = []
        train_accuracies = []
        val_accuracies = []
        val_precisions = []
        val_recalls = []
        val_f1_scores = []
        
        # Initialize best metrics for this fold
        fold_best_metric = {'epoch': -1, 'val_acc': 0, 'val_loss': float('inf')}
        
        # Train the model
        for epoch in range(CONFIG['num_epochs']):
            # Training
            train_loss, train_acc, train_prec, train_rec, train_f1 = train_epoch(
                model, train_loader, optimizer, loss_function, epoch
            )
            
            # Validation
            val_loss, val_acc, val_prec, val_rec, val_f1, val_labels, val_scores = validate_epoch(
                model, val_loader, loss_function, epoch
            )
            
            # Record history
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accuracies.append(train_acc)
            val_accuracies.append(val_acc)
            val_precisions.append(val_prec)
            val_recalls.append(val_rec)
            val_f1_scores.append(val_f1)
            
            print(f"Epoch {epoch+1}/{CONFIG['num_epochs']}: "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
                  f"Val Precision: {val_prec:.4f}, Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}")
            
            # Save model if it's the best so far for this fold
            if val_acc > fold_best_metric['val_acc'] or (val_acc == fold_best_metric['val_acc'] and val_loss < fold_best_metric['val_loss']):
                fold_best_metric = {
                    'epoch': epoch,
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'val_precision': val_prec,
                    'val_recall': val_rec,
                    'val_f1': val_f1
                }
                
                model_path = os.path.join(CONFIG['model_save_dir'], f"{run_name}_fold{fold_idx+1}_best.pth")
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'train_acc': train_acc,
                    'train_loss': train_loss
                }, model_path)
                print(f"Saved best model for fold {fold_idx+1} at epoch {epoch+1}")
            
            # Update global best if this fold is better
            if val_acc > best_fold_metrics['val_acc'] or (val_acc == best_fold_metrics['val_acc'] and val_loss < best_fold_metrics['val_loss']):
                best_fold_metrics = {
                    'fold_idx': fold_idx,
                    'epoch': epoch,
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'val_precision': val_prec,
                    'val_recall': val_rec,
                    'val_f1': val_f1
                }
        
        # Store the best metrics for this fold
        fold_best_metrics.append(fold_best_metric)
        
        # Save fold results
        fold_results = {
            'fold_idx': fold_idx,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies,
            'val_precisions': val_precisions,
            'val_recalls': val_recalls,
            'val_f1_scores': val_f1_scores,
            'best_epoch': fold_best_metric['epoch'],
            'best_val_acc': fold_best_metric['val_acc'],
            'best_val_loss': fold_best_metric['val_loss']
        }
        all_fold_results.append(fold_results)
        
        # Calculate ROC curve for this fold and store for later
        fold_results['fpr'], fold_results['tpr'], _ = roc_curve(val_labels, val_scores)
        fold_results['roc_auc'] = auc(fold_results['fpr'], fold_results['tpr'])
        
        # Plot confusion matrix for the last epoch - just store it, don't save individual plots
        cm = confusion_matrix(val_labels, np.round(val_scores))
        fold_results['val_confusion_matrix'] = cm
    
    # Calculate average results across all folds
    avg_best_val_acc = np.mean([fold['best_val_acc'] for fold in all_fold_results])
    avg_best_val_loss = np.mean([fold['best_val_loss'] for fold in all_fold_results])
    
    print("\nCross-validation completed!")
    print(f"Average best validation accuracy across folds: {avg_best_val_acc:.4f}")
    print(f"Average best validation loss across folds: {avg_best_val_loss:.4f}")
    print(f"Best fold: {best_fold_metrics['fold_idx']+1} with accuracy {best_fold_metrics['val_acc']:.4f}")
    
    # Plot combined cross-validation results
    plot_cv_results(all_fold_results, save_dir=CONFIG['graphs_dir'], timestamp=timestamp)
    
    # Plot combined ROC curves from all folds
    plt.figure(figsize=(10, 8))
    
    # Calculate average ROC curve across folds
    # Interpolating at standard FPR points to make averaging possible
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    aucs = []

    # Color maps for consistent colors across plots
    fold_colors = plt.cm.tab10(np.linspace(0, 1, CONFIG['n_folds']))  # Fixed: Use CONFIG['n_folds'] instead of num_folds
    
    # Plot individual fold ROC curves with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        fpr = fold_results['fpr']
        tpr = fold_results['tpr']
        roc_auc = fold_results['roc_auc']
        
        plt.plot(fpr, tpr, lw=1, alpha=0.3, color=fold_colors[fold_idx],
                 label=f'Fold {fold_idx+1} ROC (AUC = {roc_auc:.2f})')
        
        # Interpolate TPR values at the standard FPR points for averaging
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(roc_auc)
    
    # Calculate and plot the mean ROC curve
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    plt.plot(mean_fpr, mean_tpr, color='blue', lw=2, 
             label=f'Mean ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})')
    
    # Standard deviation band around the mean ROC
    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=0.2,
                     label=f'± 1 std. dev.')
    
    # Reference diagonal line
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Cross-Validation ROC Curves')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['graphs_dir'], f"cv_roc_curves_{timestamp}.pdf"))
    plt.close()
    
    end_time = timer()
    print(f"Cross-validation completed in {end_time - start_time:.2f} seconds")
    
    return all_fold_results

# Plot CV Results

In [10]:
def plot_cv_results(all_fold_results, save_dir="graphs", timestamp=None):
    """Plot combined results from cross-validation."""
    # Create timestamp if not provided
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Create directory for graphs if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Number of folds
    num_folds = len(all_fold_results)
    
    # Number of epochs
    epochs = len(all_fold_results[0]['train_losses'])
    epoch_nums = list(range(1, epochs + 1))
    
    # Color maps for consistent colors across plots
    fold_colors = plt.cm.tab10(np.linspace(0, 1, num_folds))
    
    # 1. Training Loss Plot
    plt.figure(figsize=(10, 6))
    
    # Calculate average training loss across folds
    avg_train_loss = np.mean([fold['train_losses'] for fold in all_fold_results], axis=0)
    
    # Plot individual fold lines with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        plt.plot(epoch_nums, fold_results['train_losses'], 
                 color=fold_colors[fold_idx], alpha=0.3, 
                 label=f'Fold {fold_idx+1}')
    
    # Plot average with full opacity and thicker line
    plt.plot(epoch_nums, avg_train_loss, color='blue', linewidth=2.5, 
             label='Average across folds')
    
    plt.title('Training Loss by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/training_loss_{timestamp}.pdf")
    plt.close()
    
    # 2. Validation Loss Plot
    plt.figure(figsize=(10, 6))
    
    # Calculate average validation loss across folds
    avg_val_loss = np.mean([fold['val_losses'] for fold in all_fold_results], axis=0)
    
    # Plot individual fold lines with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        plt.plot(epoch_nums, fold_results['val_losses'], 
                 color=fold_colors[fold_idx], alpha=0.3, 
                 label=f'Fold {fold_idx+1}')
    
    # Plot average with full opacity and thicker line
    plt.plot(epoch_nums, avg_val_loss, color='red', linewidth=2.5, 
             label='Average across folds')
    
    plt.title('Validation Loss by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/validation_loss_{timestamp}.pdf")
    plt.close()
    
    # 3. Validation Accuracy Plot
    plt.figure(figsize=(10, 6))
    
    # Calculate average validation accuracy across folds
    avg_val_acc = np.mean([fold['val_accuracies'] for fold in all_fold_results], axis=0)
    
    # Plot individual fold lines with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        plt.plot(epoch_nums, fold_results['val_accuracies'], 
                 color=fold_colors[fold_idx], alpha=0.3, 
                 label=f'Fold {fold_idx+1}')
    
    # Plot average with full opacity and thicker line
    plt.plot(epoch_nums, avg_val_acc, color='green', linewidth=2.5, 
             label='Average across folds')
    
    plt.title('Validation Accuracy by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/validation_accuracy_{timestamp}.pdf")
    plt.close()
    
    # 4. Validation Precision and Recall Plot
    plt.figure(figsize=(10, 6))
    
    # Calculate average metrics across folds
    avg_precision = np.mean([fold['val_precisions'] for fold in all_fold_results], axis=0)
    avg_recall = np.mean([fold['val_recalls'] for fold in all_fold_results], axis=0)
    
    # Plot individual fold precision lines with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        plt.plot(epoch_nums, fold_results['val_precisions'], 
                 color=fold_colors[fold_idx], alpha=0.15, 
                 linestyle='-', linewidth=1)
        plt.plot(epoch_nums, fold_results['val_recalls'], 
                 color=fold_colors[fold_idx], alpha=0.15, 
                 linestyle='--', linewidth=1)
    
    # Plot average precision and recall with full opacity
    plt.plot(epoch_nums, avg_precision, color='purple', linewidth=2.5, 
             label='Avg Precision')
    plt.plot(epoch_nums, avg_recall, color='orange', linewidth=2.5, 
             linestyle='--', label='Avg Recall')
    
    plt.title('Validation Precision and Recall by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/validation_precision_recall_{timestamp}.pdf")
    plt.close()
    
    # 5. Validation F1 Score Plot
    plt.figure(figsize=(10, 6))
    
    # Calculate average F1 score across folds
    avg_f1 = np.mean([fold['val_f1_scores'] for fold in all_fold_results], axis=0)
    
    # Plot individual fold lines with lower opacity
    for fold_idx, fold_results in enumerate(all_fold_results):
        plt.plot(epoch_nums, fold_results['val_f1_scores'], 
                 color=fold_colors[fold_idx], alpha=0.3, 
                 label=f'Fold {fold_idx+1}')
    
    # Plot average with full opacity and thicker line
    plt.plot(epoch_nums, avg_f1, color='brown', linewidth=2.5, 
             label='Average across folds')
    
    plt.title('Validation F1 Score by Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/validation_f1_{timestamp}.pdf")
    plt.close()

# Test Best Model from CV

In [11]:
def test_best_model():
    """Test the best model from cross-validation on the test set."""
    global best_fold_metrics
    
    print("\nTesting the best model on the test set...")
    
    # Get all patient IDs used in training
    train_data_dicts = create_data_dicts(CONFIG['train_dir'])
    train_patient_ids = set(d['patient_id'] for d in train_data_dicts)
    print(f"Number of unique patients in training set: {len(train_patient_ids)}")
    
    # Get test data and check for patient overlap
    test_data_dicts = create_data_dicts(CONFIG['test_dir'])
    test_patient_ids = set(d['patient_id'] for d in test_data_dicts)
    print(f"Number of unique patients in test set: {len(test_patient_ids)}")
    
    # Check for overlap
    overlap = train_patient_ids.intersection(test_patient_ids)
    if overlap:
        print(f"WARNING: Found {len(overlap)} patients in both train and test sets!")
        print(f"Overlapping patient IDs: {overlap}")
    else:
        print("No patient overlap found between train and test sets. Good!")
    
    # Load the best model from cross-validation
    best_fold = best_fold_metrics['fold_idx']
    model_path = os.path.join(CONFIG['model_save_dir'], f"{run_name}_fold{best_fold+1}_best.pth")
    
    # Initialize model
    model = initialize_model()
    
    # Load model state
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Define transforms
    transforms = Compose([
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        Orientationd(keys=["image"], axcodes="RAS"),
        Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
        #CenterSpatialCropd(keys=["image"], roi_size=CONFIG['roi_size']),
        ResizeD(keys=["image"], spatial_size=CONFIG['resize_d'], mode="nearest"),
        ScaleIntensityd(keys=["image"]),
        ToTensord(keys=["image"]),
    ])
    
    # Create test data dictionaries
    test_dicts = create_data_dicts(CONFIG['test_dir'])
    test_ds = Dataset(data=test_dicts, transform=transforms)
    test_loader = DataLoader(
        test_ds,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=torch.cuda.is_available()
    )
    
    # Test all fold models on the test set
    all_fold_test_results = []
    fold_test_cms = []
    
    # Test each fold's best model
    for fold_idx in range(CONFIG['n_folds']):
        fold_model_path = os.path.join(CONFIG['model_save_dir'], f"{run_name}_fold{fold_idx+1}_best.pth")
        
        # Skip if model doesn't exist
        if not os.path.exists(fold_model_path):
            print(f"Model for fold {fold_idx+1} not found, skipping...")
            continue
            
        # Initialize a new model and load weights
        fold_model = initialize_model()
        checkpoint = torch.load(fold_model_path)
        fold_model.load_state_dict(checkpoint['model_state_dict'])
        fold_model.eval()
        
        # Test the fold model
        y_pred = []
        y_true = []
        y_scores = []
        
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Testing fold {fold_idx+1}", unit="batch"):
                inputs, labels = batch["image"].to(device), batch["label"].to(device)
                
                outputs = fold_model(inputs)
                
                # Get predictions and scores
                probs = torch.softmax(outputs, dim=1)
                scores = probs[:, 1].cpu().numpy()  # Probability for class 1 (AD)
                _, predicted = torch.max(outputs, 1)
                
                y_pred.extend(predicted.cpu().numpy())
                y_true.extend(labels.cpu().numpy())
                y_scores.extend(scores)
        
        # Calculate metrics
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='binary')
        recall = recall_score(y_true, y_pred, average='binary')
        f1 = f1_score(y_true, y_pred, average='binary')
        
        # Calculate ROC and AUC
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        
        # Calculate confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        fold_test_cms.append(cm)
        
        # Store results for this fold
        fold_test_results = {
            'fold_idx': fold_idx,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'fpr': fpr,
            'tpr': tpr,
            'y_true': y_true,
            'y_pred': y_pred,
            'y_scores': y_scores,
            'confusion_matrix': cm
        }
        
        all_fold_test_results.append(fold_test_results)
        
        print(f"Fold {fold_idx+1} Test Results:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1 Score: {f1:.4f}")
        print(f"  ROC AUC: {roc_auc:.4f}")
    
    # Calculate average metrics across all folds
    avg_accuracy = np.mean([fold['accuracy'] for fold in all_fold_test_results])
    avg_precision = np.mean([fold['precision'] for fold in all_fold_test_results])
    avg_recall = np.mean([fold['recall'] for fold in all_fold_test_results])
    avg_f1 = np.mean([fold['f1'] for fold in all_fold_test_results])
    avg_roc_auc = np.mean([fold['roc_auc'] for fold in all_fold_test_results])
    
    # Calculate average confusion matrix
    avg_cm = np.mean(fold_test_cms, axis=0).astype(int)
    
    # Get results from the best overall model
    best_fold_idx = best_fold_metrics['fold_idx']
    best_fold_test_results = next((r for r in all_fold_test_results if r['fold_idx'] == best_fold_idx), None)
    
    # If best fold wasn't included in tests for some reason, use the model we already loaded
    if best_fold_test_results is None:
        print(f"\nBest fold {best_fold+1} results not found in test results, evaluating separately...")
        # Test the best model
        with torch.no_grad():
            y_pred = []
            y_true = []
            y_scores = []
            
            for batch in tqdm(test_loader, desc="Testing best model", unit="batch"):
                inputs, labels = batch["image"].to(device), batch["label"].to(device)
                
                outputs = model(inputs)
                
                # Get predictions and scores
                probs = torch.softmax(outputs, dim=1)
                scores = probs[:, 1].cpu().numpy()
                _, predicted = torch.max(outputs, 1)
                
                y_pred.extend(predicted.cpu().numpy())
                y_true.extend(labels.cpu().numpy())
                y_scores.extend(scores)
        
        # Calculate metrics for best model
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='binary')
        recall = recall_score(y_true, y_pred, average='binary')
        f1 = f1_score(y_true, y_pred, average='binary')
        
        # Calculate ROC and AUC
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        
        best_fold_test_results = {
            'fold_idx': best_fold,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'fpr': fpr,
            'tpr': tpr,
            'y_true': y_true,
            'y_pred': y_pred,
            'y_scores': y_scores,
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
    else:
        print(f"\nBest fold {best_fold+1} results already included in test results.")
    
    print("\nAverage Test Results Across All Folds:")
    print(f"  Accuracy: {avg_accuracy:.4f}")
    print(f"  Precision: {avg_precision:.4f}")
    print(f"  Recall: {avg_recall:.4f}")
    print(f"  F1 Score: {avg_f1:.4f}")
    print(f"  ROC AUC: {avg_roc_auc:.4f}")
    
    # Plot combined ROC curves from all fold tests
    plt.figure(figsize=(10, 8))
    
    # Calculate average ROC curve across folds
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    aucs = []
    
    fold_colors = plt.cm.tab10(np.linspace(0, 1, len(all_fold_test_results)))
    
    # Plot individual fold ROC curves with lower opacity
    for i, fold_results in enumerate(all_fold_test_results):
        fpr = fold_results['fpr']
        tpr = fold_results['tpr']
        roc_auc = fold_results['roc_auc']
        
        plt.plot(fpr, tpr, lw=1, alpha=0.3, color=fold_colors[i],
                 label=f'Fold {fold_results["fold_idx"]+1} (AUC = {roc_auc:.2f})')
        
        # Interpolate TPR values at the standard FPR points for averaging
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(roc_auc)
    
    # Calculate and plot the mean ROC curve
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    plt.plot(mean_fpr, mean_tpr, color='blue', lw=2, 
             label=f'Mean ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})')
    
    # Standard deviation band around the mean ROC
    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=0.2,
                     label=f'± 1 std. dev.')
    
    # Highlight best model ROC if desired
    if best_fold_test_results is not None:
        plt.plot(best_fold_test_results['fpr'], best_fold_test_results['tpr'], 
                 color='red', lw=1.5, linestyle='--',
                 label=f'Best model (Fold {best_fold+1}, AUC = {best_fold_test_results["roc_auc"]:.2f})')
    
    # Reference diagonal line
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Test Set ROC Curves (All Folds)')
    plt.legend(loc="lower right")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['graphs_dir'], f"{run_name}_test_all_folds_roc.pdf"))
    plt.close()
    
    # Plot average confusion matrix
    plt.figure(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=avg_cm, display_labels=CONFIG['class_names'])
    disp.plot(cmap=plt.cm.Blues)
    plt.title('Average Test Set Confusion Matrix Across All Folds')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['graphs_dir'], f"{run_name}_avg_test_cm.pdf"))
    plt.close()
    
    # Create subplot figure with all fold confusion matrices
    n_folds = len(all_fold_test_results)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))  # 2x3 grid for 5 folds + average
    axes = axes.flatten()
    
    # Plot each fold's confusion matrix
    for i, fold_results in enumerate(all_fold_test_results):
        ax = axes[i]
        cm = fold_results['confusion_matrix']
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CONFIG['class_names'])
        disp.plot(ax=ax, cmap=plt.cm.Blues, values_format='d')
        ax.set_title(f'Fold {fold_results["fold_idx"]+1} Test CM')
    
    # Plot the average confusion matrix in the last position
    ax = axes[-1]
    disp = ConfusionMatrixDisplay(confusion_matrix=avg_cm, display_labels=CONFIG['class_names'])
    disp.plot(ax=ax, cmap=plt.cm.Blues, values_format='d')
    ax.set_title(f'Average Test CM')
    
    # Hide any unused subplots
    for i in range(n_folds + 1, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['graphs_dir'], f"{run_name}_all_folds_cm.pdf"))
    plt.close()
    
    # Save test results to file
    test_results = {
        'best_model': best_fold_test_results,
        'average_across_folds': {
            'accuracy': avg_accuracy,
            'precision': avg_precision,
            'recall': avg_recall,
            'f1': avg_f1,
            'roc_auc': avg_roc_auc,
            'confusion_matrix': avg_cm.tolist()
        },
        'all_fold_results': [
            {
                'fold_idx': f['fold_idx'],
                'accuracy': f['accuracy'],
                'precision': f['precision'],
                'recall': f['recall'],
                'f1': f['f1'],
                'roc_auc': f['roc_auc']
            } for f in all_fold_test_results
        ]
    }
    
    with open(os.path.join(CONFIG['results_dir'], f"{run_name}_test_results.txt"), 'w') as f:
        f.write(f"Model: {run_name}\n\n")
        f.write("BEST MODEL RESULTS:\n")
        f.write(f"Best Fold: {best_fold+1}\n")
        f.write(f"Best Epoch: {best_fold_test_results.get('best_epoch', 'N/A')}\n")
        f.write(f"Test Accuracy: {best_fold_test_results['accuracy']:.4f}\n")
        f.write(f"Test Precision: {best_fold_test_results['precision']:.4f}\n")
        f.write(f"Test Recall: {best_fold_test_results['recall']:.4f}\n")
        f.write(f"Test F1 Score: {best_fold_test_results['f1']:.4f}\n")
        f.write(f"Test ROC AUC: {best_fold_test_results['roc_auc']:.4f}\n\n")
        
        f.write("AVERAGE RESULTS ACROSS ALL FOLDS:\n")
        f.write(f"Average Test Accuracy: {avg_accuracy:.4f}\n")
        f.write(f"Average Test Precision: {avg_precision:.4f}\n")
        f.write(f"Average Test Recall: {avg_recall:.4f}\n")
        f.write(f"Average Test F1 Score: {avg_f1:.4f}\n")
        f.write(f"Average Test ROC AUC: {avg_roc_auc:.4f}\n")
    
    return test_results

# Summarize Results

In [12]:
def summarize_results(test_results):
    """Print a summary of the best model's performance."""
    print("\n" + "="*50)
    print(f"SUMMARY OF RESULTS FOR {run_name}")
    print("="*50)
    
    print("BEST MODEL RESULTS:")
    print(f"Best model from fold {test_results['best_model']['fold_idx']+1}")
    print(f"Test Accuracy: {test_results['best_model']['accuracy']:.4f}")
    print(f"Test Precision: {test_results['best_model']['precision']:.4f}")
    print(f"Test Recall: {test_results['best_model']['recall']:.4f}")
    print(f"Test F1 Score: {test_results['best_model']['f1']:.4f}")
    print(f"Test ROC AUC: {test_results['best_model']['roc_auc']:.4f}")
    
    print("\nAVERAGE RESULTS ACROSS ALL FOLDS:")
    print(f"Average Test Accuracy: {test_results['average_across_folds']['accuracy']:.4f}")
    print(f"Average Test Precision: {test_results['average_across_folds']['precision']:.4f}")
    print(f"Average Test Recall: {test_results['average_across_folds']['recall']:.4f}")
    print(f"Average Test F1 Score: {test_results['average_across_folds']['f1']:.4f}")
    print(f"Average Test ROC AUC: {test_results['average_across_folds']['roc_auc']:.4f}")
    print("="*50)

# Save data

In [13]:
def save_results_to_csv(all_fold_results, test_results, save_dir="results", timestamp=None):
    """Save all training history and test results to CSV files for later analysis."""
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    os.makedirs(save_dir, exist_ok=True)
    
    # Save experiment configuration
    with open(os.path.join(save_dir, f"{run_name}_config.json"), 'w') as f:
        # Create a copy of CONFIG that's JSON serializable
        config_to_save = {k: v for k, v in CONFIG.items() if not callable(v)}
        config_to_save['experiment_timestamp'] = timestamp
        config_to_save['run_name'] = run_name
        json.dump(config_to_save, f, indent=4)
    
    # 1. Save training history for all folds
    for fold_idx, fold_results in enumerate(all_fold_results):
        history_data = {
            'epoch': list(range(1, len(fold_results['train_losses']) + 1)),
            'train_loss': fold_results['train_losses'],
            'val_loss': fold_results['val_losses'],
            'train_acc': fold_results['train_accuracies'],
            'val_acc': fold_results['val_accuracies'],
            'val_precision': fold_results['val_precisions'],
            'val_recall': fold_results['val_recalls'],
            'val_f1': fold_results['val_f1_scores']
        }
        
        # Save to CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(os.path.join(save_dir, f"{run_name}_fold{fold_idx+1}_history.csv"), index=False)
    
    # 2. Save ROC curve data for training folds
    for fold_idx, fold_results in enumerate(all_fold_results):
        if 'fpr' in fold_results and 'tpr' in fold_results:
            # Save to CSV
            roc_df = pd.DataFrame({'fpr': fold_results['fpr'], 'tpr': fold_results['tpr']})
            roc_df.to_csv(os.path.join(save_dir, f"{run_name}_fold{fold_idx+1}_val_roc.csv"), index=False)
    
    # 3. Save all fold metrics in a combined CSV for easier comparison
    all_folds_val_metrics = []
    for fold_idx, fold_results in enumerate(all_fold_results):
        metrics = {
            'fold_idx': fold_idx,
            'best_epoch': fold_results['best_epoch'],
            'best_val_acc': fold_results['best_val_acc'],
            'best_val_loss': fold_results['best_val_loss']
        }
        all_folds_val_metrics.append(metrics)
    
    all_folds_df = pd.DataFrame(all_folds_val_metrics)
    all_folds_df.to_csv(os.path.join(save_dir, f"{run_name}_all_folds_val_metrics.csv"), index=False)
    
    # 4. Save best model test results if available
    if 'best_model' in test_results:
        best_model = test_results['best_model']
        best_model_metrics = {}
        
        # Extract all available metrics
        for key in ['fold_idx', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
            if key in best_model:
                best_model_metrics[key] = best_model[key]
        
        # Save metrics to JSON
        with open(os.path.join(save_dir, f"{run_name}_best_model_metrics.json"), 'w') as f:
            json.dump(best_model_metrics, f, indent=4)
        
        # Save ROC data if available
        if 'fpr' in best_model and 'tpr' in best_model:
            # Convert to list if numpy array
            fpr = best_model['fpr'].tolist() if isinstance(best_model['fpr'], np.ndarray) else best_model['fpr']
            tpr = best_model['tpr'].tolist() if isinstance(best_model['tpr'], np.ndarray) else best_model['tpr']
            
            best_roc_df = pd.DataFrame({
                'fpr': fpr,
                'tpr': tpr
            })
            best_roc_df.to_csv(os.path.join(save_dir, f"{run_name}_best_model_roc.csv"), index=False)
        
        # Save full predictions and ground truth if available
        if 'y_true' in best_model and 'y_pred' in best_model and 'y_scores' in best_model:
            predictions_df = pd.DataFrame({
                'y_true': best_model['y_true'],
                'y_pred': best_model['y_pred'],
                'y_score': best_model['y_scores']
            })
            predictions_df.to_csv(os.path.join(save_dir, f"{run_name}_best_model_predictions.csv"), index=False)
    
    # 5. Save average test metrics
    if 'average_across_folds' in test_results:
        avg_metrics = test_results['average_across_folds']
        # Filter out confusion matrix which isn't serializable directly
        avg_metrics_filtered = {k: v for k, v in avg_metrics.items() if k != 'confusion_matrix'}
        
        with open(os.path.join(save_dir, f"{run_name}_avg_metrics.json"), 'w') as f:
            json.dump(avg_metrics_filtered, f, indent=4)
    
    # 6. Save per-fold test metrics if they exist
    if 'all_fold_results' in test_results:
        test_df = pd.DataFrame(test_results['all_fold_results'])
        test_df.to_csv(os.path.join(save_dir, f"{run_name}_all_folds_test_metrics.csv"), index=False)
    
    # 7. Save confusion matrices if available
    if 'best_model' in test_results and 'confusion_matrix' in test_results['best_model']:
        matrices = {}
        
        # Handle best model confusion matrix
        if 'confusion_matrix' in test_results['best_model']:
            cm = test_results['best_model']['confusion_matrix']
            matrices['best_model'] = cm.tolist() if isinstance(cm, np.ndarray) else cm
        
        # Handle average confusion matrix if available
        if 'average_across_folds' in test_results and 'confusion_matrix' in test_results['average_across_folds']:
            avg_cm = test_results['average_across_folds']['confusion_matrix']
            matrices['average'] = avg_cm if isinstance(avg_cm, list) else avg_cm.tolist()
        
        # Save individual fold confusion matrices
        if 'all_fold_results' in test_results:
            for i, fold_result in enumerate(test_results['all_fold_results']):
                if 'confusion_matrix' in fold_result:
                    cm = fold_result['confusion_matrix']
                    matrices[f'fold_{fold_result["fold_idx"]}'] = cm.tolist() if isinstance(cm, np.ndarray) else cm
        
        # Save if we have matrices
        if matrices:
            with open(os.path.join(save_dir, f"{run_name}_confusion_matrices.json"), 'w') as f:
                json.dump(matrices, f, indent=4)
    
    print(f"All results saved to CSV/JSON files in {save_dir} directory")


# Run

In [14]:
print(f"Starting {run_name} experiment...")

# Analyze patient distribution
analyze_patient_distribution()

# Run cross-validation
all_fold_results = run_cross_validation()
    
# Test the best model
test_results = test_best_model()
    
# Summarize results
summarize_results(test_results)

# Save all results to CSV for later analysis
save_results_to_csv(all_fold_results, test_results, save_dir=CONFIG['results_dir'], timestamp=timestamp)

Starting densenet201_20250326_155643 experiment...
Total unique patients in training: 280
Total unique patients in testing: 48
Patient overlap between train and test: 0
Patient distribution saved to results\densenet201_20250326_155643_patient_distribution.csv
Starting 5-fold cross-validation...
Total samples in training dataset: 840
Total unique patients: 280
Class CN (label 0): 498 samples from 166 unique patients
Class AD (label 1): 342 samples from 114 unique patients

Training Fold 1/5
Train size: 672 scans from 224 patients
Validation size: 168 scans from 56 patients
Train class distribution:
  CN: 393 samples (58.5%)
  AD: 279 samples (41.5%)
Validation class distribution:
  CN: 105 samples (62.5%)
  AD: 63 samples (37.5%)


Training epoch 1:   0%|          | 0/84 [00:00<?, ?batch/s]

Validation epoch 1:   0%|          | 0/21 [00:00<?, ?batch/s]

Epoch 1/20: Train Loss: 0.7159, Train Acc: 0.5655, Val Loss: 0.8647, Val Acc: 0.5238, Val Precision: 0.3607, Val Recall: 0.3492, Val F1: 0.3548
Saved best model for fold 1 at epoch 1


Training epoch 2:   0%|          | 0/84 [00:00<?, ?batch/s]

...

Testing fold 4:   0%|          | 0/18 [00:00<?, ?batch/s]

Fold 4 Test Results:
  Accuracy: 0.7500
  Precision: 0.8387
  Recall: 0.4561
  F1 Score: 0.5909
  ROC AUC: 0.8538


Testing fold 5:   0%|          | 0/18 [00:00<?, ?batch/s]

Fold 5 Test Results:
  Accuracy: 0.7778
  Precision: 0.6667
  Recall: 0.8772
  F1 Score: 0.7576
  ROC AUC: 0.8639

Best fold 2 results already included in test results.

Average Test Results Across All Folds:
  Accuracy: 0.7597
  Precision: 0.7672
  Recall: 0.6000
  F1 Score: 0.6479
  ROC AUC: 0.8599

SUMMARY OF RESULTS FOR densenet201_20250326_155643
BEST MODEL RESULTS:
Best model from fold 2
Test Accuracy: 0.7292
Test Precision: 0.8214
Test Recall: 0.4035
Test F1 Score: 0.5412
Test ROC AUC: 0.8693

AVERAGE RESULTS ACROSS ALL FOLDS:
Average Test Accuracy: 0.7597
Average Test Precision: 0.7672
Average Test Recall: 0.6000
Average Test F1 Score: 0.6479
Average Test ROC AUC: 0.8599
All results saved to CSV/JSON files in results directory


<Figure size 800x600 with 0 Axes>

# recreate graphs

In [43]:
def recreate_graphs(run_name, results_dir="results", graphs_dir="graphs"):
    """
    Recreate all graphs for a specific run using saved CSV and JSON data.
    
    Parameters:
    -----------
    run_name : str
        Name of the run/experiment to recreate graphs for
    results_dir : str
        Directory containing the saved results
    graphs_dir : str
        Directory to save the recreated graphs
    """
    print(f"Recreating graphs for run: {run_name}")
    
    # Create graphs directory if it doesn't exist
    os.makedirs(graphs_dir, exist_ok=True)
    
    # Load configuration
    config_path = os.path.join(results_dir, f"{run_name}_config.json")
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"Loaded configuration for run: {run_name}")
        
        # Extract timestamp if available, otherwise use run_name
        timestamp = config.get('experiment_timestamp', run_name)
        class_names = config.get('class_names', ['CN', 'AD'])  # Default if not found
        n_folds = config.get('n_folds', 5)  # Default if not found
    else:
        print(f"Warning: Configuration file not found. Using default values.")
        timestamp = run_name
        class_names = ['CN', 'AD']
        n_folds = 5
    
    # 1. Load training history for all folds
    all_fold_results = []
    for fold_idx in range(n_folds):
        fold_history_path = os.path.join(results_dir, f"{run_name}_fold{fold_idx+1}_history.csv")
        if os.path.exists(fold_history_path):
            fold_df = pd.read_csv(fold_history_path)
            
            # Reconstruct fold results dictionary
            fold_results = {
                'fold_idx': fold_idx,
                'train_losses': fold_df['train_loss'].tolist(),
                'val_losses': fold_df['val_loss'].tolist(),
                'train_accuracies': fold_df['train_acc'].tolist(),
                'val_accuracies': fold_df['val_acc'].tolist(),
                'val_precisions': fold_df['val_precision'].tolist(),
                'val_recalls': fold_df['val_recall'].tolist(),
                'val_f1_scores': fold_df['val_f1'].tolist()
            }
            
            # Load ROC data if available
            fold_roc_path = os.path.join(results_dir, f"{run_name}_fold{fold_idx+1}_val_roc.csv")
            if os.path.exists(fold_roc_path):
                roc_df = pd.read_csv(fold_roc_path)
                fold_results['fpr'] = roc_df['fpr'].values
                fold_results['tpr'] = roc_df['tpr'].values
                fold_results['roc_auc'] = auc(fold_results['fpr'], fold_results['tpr'])
            
            # Load best metrics for this fold if available
            fold_metrics_path = os.path.join(results_dir, f"{run_name}_all_folds_val_metrics.csv")
            if os.path.exists(fold_metrics_path):
                all_folds_df = pd.read_csv(fold_metrics_path)
                fold_row = all_folds_df[all_folds_df['fold_idx'] == fold_idx]
                if not fold_row.empty:
                    fold_results['best_epoch'] = fold_row['best_epoch'].values[0]
                    fold_results['best_val_acc'] = fold_row['best_val_acc'].values[0]
                    fold_results['best_val_loss'] = fold_row['best_val_loss'].values[0]
            
            all_fold_results.append(fold_results)
    
    # 2. Load test results
    test_results = {}
    
    # Load best model metrics
    best_model_path = os.path.join(results_dir, f"{run_name}_best_model_metrics.json")
    if os.path.exists(best_model_path):
        with open(best_model_path, 'r') as f:
            best_model_metrics = json.load(f)
        
        # Initialize best_model dictionary
        test_results['best_model'] = best_model_metrics
        
        # Load ROC data if available
        best_roc_path = os.path.join(results_dir, f"{run_name}_best_model_roc.csv")
        if os.path.exists(best_roc_path):
            roc_df = pd.read_csv(best_roc_path)
            test_results['best_model']['fpr'] = roc_df['fpr'].values
            test_results['best_model']['tpr'] = roc_df['tpr'].values
        
        # Load predictions if available
        best_preds_path = os.path.join(results_dir, f"{run_name}_best_model_predictions.csv")
        if os.path.exists(best_preds_path):
            preds_df = pd.read_csv(best_preds_path)
            test_results['best_model']['y_true'] = preds_df['y_true'].values
            test_results['best_model']['y_pred'] = preds_df['y_pred'].values
            test_results['best_model']['y_scores'] = preds_df['y_score'].values
    
    # Load average metrics
    avg_metrics_path = os.path.join(results_dir, f"{run_name}_avg_metrics.json")
    if os.path.exists(avg_metrics_path):
        with open(avg_metrics_path, 'r') as f:
            avg_metrics = json.load(f)
        test_results['average_across_folds'] = avg_metrics
    
    # Load all folds test metrics
    all_folds_test_path = os.path.join(results_dir, f"{run_name}_all_folds_test_metrics.csv")
    if os.path.exists(all_folds_test_path):
        test_df = pd.read_csv(all_folds_test_path)
        test_results['all_fold_results'] = test_df.to_dict('records')
    
    # Load confusion matrices
    cm_path = os.path.join(results_dir, f"{run_name}_confusion_matrices.json")
    if os.path.exists(cm_path):
        with open(cm_path, 'r') as f:
            matrices = json.load(f)
        
        # Add confusion matrices to appropriate places in test_results
        if 'best_model' in matrices and 'best_model' in test_results:
            test_results['best_model']['confusion_matrix'] = np.array(matrices['best_model'])
        
        if 'average' in matrices:
            if 'average_across_folds' not in test_results:
                test_results['average_across_folds'] = {}
            test_results['average_across_folds']['confusion_matrix'] = np.array(matrices['average'])
    
    # 3. Now recreate the graphs if we have enough data
    if all_fold_results:
        print(f"Loaded data for {len(all_fold_results)} folds. Recreating training graphs...")
        plot_cv_results(all_fold_results, save_dir=graphs_dir, timestamp=timestamp)
    else:
        print("Warning: No fold results loaded. Cannot recreate training graphs.")
    
    # Recreate ROC curve for cross-validation
    if all_fold_results and all('fpr' in fold and 'tpr' in fold for fold in all_fold_results):
        print("Recreating cross-validation ROC curves...")
        
        plt.figure(figsize=(10, 8))
        
        # Calculate average ROC curve across folds
        mean_fpr = np.linspace(0, 1, 100)
        tprs = []
        aucs = []
        
        # Color maps for consistent colors across plots
        fold_colors = plt.cm.tab10(np.linspace(0, 1, len(all_fold_results)))
        
        # Plot individual fold ROC curves with lower opacity
        for fold_idx, fold_results in enumerate(all_fold_results):
            fpr = fold_results['fpr']
            tpr = fold_results['tpr']
            roc_auc = fold_results['roc_auc']
            
            plt.plot(fpr, tpr, lw=1, alpha=0.3, color=fold_colors[fold_idx],
                     label=f'Fold {fold_idx+1} ROC (AUC = {roc_auc:.2f})')
            
            # Interpolate TPR values at the standard FPR points for averaging
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
            aucs.append(roc_auc)
        
        # Calculate and plot the mean ROC curve
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        mean_auc = np.mean(aucs)
        std_auc = np.std(aucs)
        
        plt.plot(mean_fpr, mean_tpr, color='blue', lw=2, 
                 label=f'Mean ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})')
        
        # Standard deviation band around the mean ROC
        std_tpr = np.std(tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=0.2,
                         label=f'± 1 std. dev.')
        
        # Reference diagonal line
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Cross-Validation ROC Curves')
        plt.legend(loc="lower right")
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(graphs_dir, f"cv_roc_curves_{timestamp}.pdf"))
        plt.close()
    
    # Recreate test ROC curves if we have the data
    if 'all_fold_results' in test_results and 'best_model' in test_results and 'fpr' in test_results['best_model']:
        print("Recreating test ROC curves...")
        
        plt.figure(figsize=(10, 8))
        
        # If we have individual fold ROCs for testing, we can plot them
        if 'all_fold_results' in test_results and all('fpr' in fold for fold in test_results['all_fold_results']):
            # Plot individual fold ROC curves
            fold_colors = plt.cm.tab10(np.linspace(0, 1, len(test_results['all_fold_results'])))
            
            for i, fold_result in enumerate(test_results['all_fold_results']):
                fpr = fold_result['fpr']
                tpr = fold_result['tpr']
                roc_auc = fold_result['roc_auc']
                
                plt.plot(fpr, tpr, lw=1, alpha=0.3, color=fold_colors[i],
                         label=f'Fold {fold_result["fold_idx"]+1} (AUC = {roc_auc:.2f})')
        
        # Plot best model ROC
        best_fpr = test_results['best_model']['fpr']
        best_tpr = test_results['best_model']['tpr']
        best_auc = auc(best_fpr, best_tpr)
        
        plt.plot(best_fpr, best_tpr, color='red', lw=1.5, linestyle='--',
                 label=f'Best model (Fold {test_results["best_model"]["fold_idx"]+1}, AUC = {best_auc:.2f})')
        
        # Reference diagonal line
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Test Set ROC Curves')
        plt.legend(loc="lower right")
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(graphs_dir, f"{run_name}_test_roc.pdf"))
        plt.close()
    
    # Plot confusion matrices if available
    if 'best_model' in test_results and 'confusion_matrix' in test_results['best_model']:
        print("Recreating confusion matrices...")
        
        # Plot best model confusion matrix
        plt.figure(figsize=(8, 6))
        best_cm = test_results['best_model']['confusion_matrix']
        disp = ConfusionMatrixDisplay(confusion_matrix=best_cm, display_labels=class_names)
        disp.plot(cmap=plt.cm.Blues)
        plt.title(f'Best Model (Fold {test_results["best_model"]["fold_idx"]+1}) Confusion Matrix')
        plt.tight_layout()
        plt.savefig(os.path.join(graphs_dir, f"{run_name}_best_model_cm.pdf"))
        plt.close()
        
        # Plot average confusion matrix if available
        if 'average_across_folds' in test_results and 'confusion_matrix' in test_results['average_across_folds']:
            plt.figure(figsize=(8, 6))
            avg_cm = test_results['average_across_folds']['confusion_matrix']
            disp = ConfusionMatrixDisplay(confusion_matrix=avg_cm, display_labels=class_names)
            disp.plot(cmap=plt.cm.Blues)
            plt.title('Average Test Set Confusion Matrix Across All Folds')
            plt.tight_layout()
            plt.savefig(os.path.join(graphs_dir, f"{run_name}_avg_test_cm.pdf"))
            plt.close()
    
    print(f"All graphs recreated and saved to {graphs_dir}")
    return all_fold_results, test_results

In [46]:
# Later, to recreate graphs from saved data
all_fold_results, test_results = recreate_graphs(run_name="densenet201_20250324_213341")

Recreating graphs for run: densenet201_20250324_213341
Loaded configuration for run: densenet201_20250324_213341
Loaded data for 2 folds. Recreating training graphs...
Recreating cross-validation ROC curves...
Recreating test ROC curves...
Recreating confusion matrices...
All graphs recreated and saved to graphs


<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>